# Scraping West Bengal State Data with Beautiful Soup

This notebook scrapes information regarding the state of West Bengal—specifically the Chief Ministers table and State Symbols—from Wikipedia and turns the raw HTML into cleaned Pandas DataFrames.


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Request the Webpage

In [2]:
url = "https://en.wikipedia.org/wiki/Chief_Minister_of_West_Bengal"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Send request to create the response variable
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

print(f"Status Code: {response.status_code}")

Status Code: 200


In [3]:
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table")
print(f"Total tables found: {len(tables)}")

Total tables found: 14


In [4]:
for i, table in enumerate(tables):
    table_text = table.get_text(" ", strip=True)
    if "Chief Minister" in table_text or "Incumbent" in table_text:
        print(f"Table index {i}: Chief Ministers Table")
    if "State symbols" in table_text:
        print(f"Table index {i}: State Symbols Table")

Table index 0: Chief Ministers Table
Table index 2: Chief Ministers Table
Table index 6: Chief Ministers Table
Table index 12: Chief Ministers Table
Table index 12: State Symbols Table


In [5]:
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table")
print(f"Total tables found: {len(tables)}")

Total tables found: 14


## 4. Identify Target Tables

In [6]:
for i, table in enumerate(tables):
    table_text = table.get_text(" ", strip=True)
    if "Chief Minister" in table_text or "Incumbent" in table_text:
        print(f"Table index {i}: Chief Ministers Table")
    if "State symbols" in table_text:
        print(f"Table index {i}: State Symbols Table")

Table index 0: Chief Ministers Table
Table index 2: Chief Ministers Table
Table index 6: Chief Ministers Table
Table index 12: Chief Ministers Table
Table index 12: State Symbols Table


## 5. Extract Chief Ministers Table

In [10]:


# Parse Table 0 safely row by row
cm_table = tables[0]
rows = cm_table.find_all("tr")

data = []
for row in rows:
    cols = [ele.text.strip() for ele in row.find_all(["td", "th"])]
    if cols:
        data.append(cols)

# Create DataFrame without forcing header row length
df_cm = pd.DataFrame(data)

# Use the first row as columns if valid, otherwise clean directly
if len(df_cm) > 0:
    df_cm.columns = df_cm.iloc[0]
    df_cm = df_cm[1:].reset_index(drop=True)

# Clean citation marks like [1]
df_cm = df_cm.replace(r'\[.*?\]', '', regex=True).dropna(how='all')

df_cm.head()

,Chief Minister of West Bengal,NaN
0,পশ্চিমবঙ্গের মুখ্যমন্ত্রীPaścimabaṅgera Mukhya...,NaN
1,Emblem of West Bengal,NaN
2,Flag of India,NaN
3,IncumbentSuvendu Adhikarisince 9 May 2026,NaN
4,Chief Minister's OfficeGovernment of West Bengal,NaN


## 7. Display Scraped Data

We inspect the first few rows of both extracted DataFrames to confirm formatting and alignment.

In [13]:
# 1. Extract State Symbols (Table 12)
symbols_table = tables[12]
s_data = [[ele.text.strip() for ele in row.find_all(["td", "th"])] for row in symbols_table.find_all("tr")]

# 2. Create DataFrame and clean citation marks
df_symbols = pd.DataFrame(s_data)
df_symbols = df_symbols.replace(r'\[.*?\]', '', regex=True).dropna(how='all')

# 3. Display preview
print("--- State Symbols DataFrame Preview ---")
display(df_symbols.head(10))

--- State Symbols DataFrame Preview ---


,0,1,2,3,4,5,6,7,8,9,10,11
0,vteState of West Bengal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Capital: Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,State symbols,Emblem: Emblem of West Bengal\nAnthem: Banglar...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,History,Gauda Kingdom\nShashanka\nPala Empire\nSena dy...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Geography,Bengal Basin\nDarjeeling Himalayan hill region...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Governance,Chief Ministers\nGovernor\nCabinet\nLegislativ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Rights groups,Matua Mahasangha\nBangla Pokkho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Divisions and districts,Burdwan division\nArambagh (declared)\nBirbhum...,Burdwan division,Arambagh (declared)\nBirbhum\nHooghly\nPaschim...,Jalpaiguri division,Alipurduar\nCooch Behar\nDarjeeling\nJalpaigur...,Malda division,Berhampore (declared)\nDakshin Dinajpur\nJangi...,Medinipur division,Bankura\nBishnupur (declared)\nJhargram\nPasch...,Presidency division,Basirhat (declared)\nHowrah\nIchamati (declare...
8,Burdwan division,Arambagh (declared)\nBirbhum\nHooghly\nPaschim...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Jalpaiguri division,Alipurduar\nCooch Behar\nDarjeeling\nJalpaigur...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Export to CSV

In [14]:
# Save DataFrames to CSV
df_cm.to_csv("west_bengal_chief_ministers.csv", index=False)
df_symbols.to_csv("west_bengal_state_symbols.csv", index=False)

print("Saved 'west_bengal_chief_ministers.csv' and 'west_bengal_state_symbols.csv' successfully!")

Saved 'west_bengal_chief_ministers.csv' and 'west_bengal_state_symbols.csv' successfully!
